# Transformers

En este notebook aprenderemos a:
1. **Tokenizar** frases y ver cómo se realiza internamente
2. **Observar la atención** de un transformer
3. **Ver las posibles salidas** que puede tener un transformer

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def token_report(text):
    toks = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(toks)
    print(f"Texto: {text}")
    print(f"Tokens ({len(toks)}): {toks}")
    print(f"IDs (primeros 12): {ids[:12]}")
    print("-" * 60)

---

## Parte 1 - Tokenización
Los LLM y transformers necesitan realizar un proceso de "tokenización". Dividen el texto en subpalabras/tokens, el contexto y el peso de cada token se calculan después en el modelo.

In [ ]:
demo_texts = [
    "Programación",
    "Programacion",
    "The cake is a lie",
    "Hay que mover el cacharro",
    "Dónde caemos gente?"
]
for t in demo_texts:
    token_report(t)

Mientras más tokens, más coste y más longitud de secuencia. Cuesta más saber completamente el contexto.
Muchas veces el tokenizer no necesita saber las palabras completas, con conocer la raíz le sirve para saber el contexto:
- habl - o
- habl - aré
- habl - ador

Esto es muy útil en idiomas con palabras extremadamente largas, ya que permite conocer cada una de las raices de las palabras que lo componen

In [ ]:
demo_texts = [
    "Donaudampfschifffahrtsgesellschaftskapitän",
    "Çekoslavakyalılaştıramadıklarımızdan",
    "Pneumonoultramicroscopicsilicovolcanoconiosis"
]
for t in demo_texts:
    token_report(t)

---

## Ejercicio 1 - Demo con palabras largas
Ver cómo se dividen las siguientes palabras en subpartes frecuentes:
- Electroencefalografista
- Otorrinolaringólogo
- Anticonstitucionalmente

In [ ]:
tarea1_texts = [
    "Electroencefalografista",
    "Otorrinolaringólogo",
    "Anticonstitucionalmente",
]
for t in tarea1_texts:
    token_report(t)

---

## Ejercicio 2 - Demo sobre diferencias en el idioma
Algunos tokenizers pueden funcionar en varios idiomas, pero otros pueden tener fallos al ver las diferentes relaciones y calcular la atención.

Eficiencia en diferentes idiomas, ¿cuál es más eficiente?

In [ ]:
tarea2_texts = {
    "es": "Necesito dinero del banco",
    "en": "I need money from the bank",
    "tr": "Bankadan para cekmem gerekiyor",
    "de": "Ich brauche Geld von der Bank",
}

results = []
for lang, text in tarea2_texts.items():
    toks = tokenizer.tokenize(text)
    results.append((lang, text, len(toks), toks))

for lang, text, n, toks in results:
    print(f"{lang} | {n} tokens | {text}")
    print(toks)
    print("-" * 60)

---

## Ejercicio 3 - Busca y Compara 3 Tokenizers

Cada tokenizer funciona diferente según el idioma, arquitectura y datos de entrenamiento. 
**Busca 2 tokenizers que tu elijas** y compáralos.

👉 **[Hugging Face Hub - Feature Extraction Models](https://huggingface.co/models?pipeline_tag=feature-extraction)**  


### Buscar 2 modelos con estos criterios:

**Modelo optimizado para IDIOMA (no inglés):**
- Busca: `spanish`, `french`, `japanese`, `german`, `portuguese`, etc.

**Modelo GENERATIVO (GPT, LLaMA, Falcon, etc.):**
- Busca: modelos que digan "generative" o nombres como `gpt2`, `mistral`, `llama`

**Para encontrar el nombre del tokenizer**:
- Lee la tarjeta del modelo (haz clic en el nombre)
- Busca el nombre completo: puede ser `usuario/nombre` o solo `nombre-modelo` (si es oficial)
- Si falla, prueba otro en la categoría

In [ ]:
from transformers import AutoTokenizer
import pandas as pd

tarea3_tests = [
    "sudo rm -rf / --no-preserve-root",  #Comando de linux
    "https://store.steampowered.com/app/753640/Outer_Wilds/",  #URL
    "なぜそれを翻訳するのですか 💀💀💀",  #Emojis y texto japonés
    "#skillissue",  #Hashtags
    "AAAAAAA BBBBBB CCCCCC",  #Palabras largas repetidas
    "SELECT * FROM Users WHERE UserId = 105 OR 1=1;", #SQL
    "function test() { return x * 2; }", #Código JavaScript
    "𓀀𓀁𓀂𓀃𓀄" #Jeroglíficos egipcios
]

# ========== MODELO 1: BERT Multilingüe ==========
model_bert = "bert-base-multilingual-cased"

# ========== TODO: AÑADE TUS 2 MODELOS AQUÍ ==========
model_language = None      #Cambia por tu modelo de idioma específico (ej: "dccuchile/bert-base-spanish-wwm-cased")
model_generative = None    #Cambia por tu modelo generativo (ej: "gpt2")

# Diccionario de modelos a comparar
my_models = {
    "BERT Multilingüe": model_bert,
    "Modelo de Idioma": model_language,
    "Modelo Generativo": model_generative,
}

#Análisis detallado
rows = []

for category, model_name in my_models.items():
    if model_name is None:
        #Si no han añadido el modelo, marcar como pendiente
        for text in tarea3_tests:
            rows.append({
                "Modelo": category,
                "Texto": text,
                "Nº tokens": "TODO",
                "Tokens": "Debes añadir el modelo arriba"
            })
        continue
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        for text in tarea3_tests:
            tokens = tokenizer.tokenize(text)
            
            rows.append({
                "Modelo": category,
                "Texto": text,
                "Nº tokens": len(tokens),
                "Tokens": " ".join(tokens[:15]) + (" ..." if len(tokens) > 15 else "")
            })
    
    except Exception as e:
        for text in tarea3_tests:
            rows.append({
                "Modelo": category,
                "Texto": text,
                "Nº tokens": "ERROR",
                "Tokens": f"Error al cargar: {str(e)[:50]}"
            })

df = pd.DataFrame(rows)

#Crear tabla pivotada para mejor visualización
pivot_df = df.pivot(index="Texto", columns="Modelo", values=["Nº tokens", "Tokens"])

styled_df = pivot_df.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]},
])

display(styled_df)

---

## Parte 2 - Embeddings y Transformers

### ¿Cómo entienden los modelos el significado?

Después de **tokenizar** (dividir en piezas), el modelo convierte cada token en un **vector numérico**. Esto se llama **embedding**.

**La idea principal** es que las palabras con significado similar tienen embeddings parecidos.

### Arquitectura del Transformer

Los transformers son muy sencillos de utilizar en código, pero internamente tienen una arquitectura específica:

1. **Tokenización**: Convierte el texto en tokens (palabras o subpalabras)
2. **Embeddings**: Transforma los tokens en vectores numéricos
3. **Capa de atención**: Permite al modelo enfocarse en partes relevantes del texto. Calcula relaciones entre todos los tokens
4. **Capa de salida**: Produce la predicción final (clasificación, generación de texto, traducción, etc.)

### ¿Es similar a una red neuronal clásica?

Sí, pero con diferencias clave:

| Característica | Deep Learning tradicional | Transformers |
|----------------|---------------------------|--------------|
| **Tokenización** | No (trabaja con datos numéricos o imágenes directamente) | Sí (divide texto en tokens) |
| **Embeddings** | Solo en NLP clásico | Siempre (vectores para cada token) |
| **Atención** | No (usa convoluciones o recurrentes) | Sí |
| **Contexto** | Local (ventanas pequeñas) | Global (ve todas las relaciones) |
| **Salida** | Clasificación principalmente | Texto, clasificación, traducción, etc. |

La **atención** es lo más revolucionario, permite conocer qué palabras son importantes y cómo se relacionan entre sí.

---

## Parte 3 - Mecanismo de Atención

La atención es lo más revolucionario del transformer. Le permite conocer si una palabra es importante o no en el texto, y **cómo se relaciona con las demás palabras**.

### Ejemplo: La palabra "banco" tiene múltiples significados

Vamos a ver cómo el modelo entiende el contexto de "banco" en diferentes frases:
- "Necesito dinero del **banco**" → institución financiera
- "Me siento en el **banco** del parque" → asiento
- "Vi un **banco** de peces en el mar" → grupo de animales

El modelo usa **atención** para saber qué palabras son más relevantes para comprender el significado de "banco".

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import matplotlib.pyplot as plt

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

#Cargamos un modelo preentrenado para ver la atención en diferentes frases

In [ ]:
sentences = [
    "Necesito dinero del banco",
    "Me siento en el banco del parque",
    "Vi un banco de peces en el mar"
]

In [ ]:
def analyze_attention(sentence):
    """
    Extrae la atención de una frase.
    
    Parámetros:
    - sentence: frase a analizar
    
    Retorna:
    - tokens: lista de tokens
    - attention: tupla con las matrices de atención de todas las capas
    """
    inputs = tokenizer(sentence, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    attention = outputs.attentions[len(outputs.attentions)-1]
    
    print(f"Frase: {sentence}")
    print(f"Tokens: {tokens}\n")

    plot_attention(tokens, attention)
    
    return tokens, attention


def plot_attention(tokens, attention):
    """
    Visualiza la atención con líneas estilo BertViz.
    
    Parámetros:
    - tokens: lista de tokens (de get_attention)
    - attention: tupla de atenciones (de get_attention)
    """
    #Extraer atención de la capa y cabeza específica
    att_matrix = attention[0].mean(dim=0).cpu().numpy()
    
    # Filtrar tokens especiales ([CLS], [SEP])
    valid_indices = [i for i, t in enumerate(tokens) if t not in ["[CLS]", "[SEP]"]]
    filtered_tokens = [tokens[i] for i in valid_indices]
    
    # Visualización con líneas
    fig, ax = plt.subplots(figsize=(10, 8))
    
    n_tokens = len(filtered_tokens)
    left_x, right_x = 0.2, 0.8
    y_positions = np.linspace(0.9, 0.1, n_tokens)
    
    # Dibujar líneas de atención (sin auto-atención)
    for idx_i, i in enumerate(valid_indices):
        for idx_j, j in enumerate(valid_indices):
            if i == j:  # Saltar auto-atención
                continue
            weight = att_matrix[i,j]
            weight = weight / att_matrix.max()  #normaliza entre 0 y 1
            if weight > 0.05:
                ax.plot([left_x, right_x], 
                       [y_positions[idx_i], y_positions[idx_j]], 
                       color='blue', alpha=weight, linewidth=weight*5)
    
    # Dibujar tokens
    for idx, token in enumerate(filtered_tokens):
        ax.text(left_x - 0.05, y_positions[idx], token, ha='right', va='center', fontsize=11, fontweight='bold')
        ax.text(right_x + 0.05, y_positions[idx], token, ha='left', va='center', fontsize=11, fontweight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(f'Atención', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
#Analizamos cada frase para ver cómo cambia la atención de "banco"
for s in sentences:
    tokens, att = analyze_attention(s)
    print("=" * 60)

Se ve que casi siempre, cada token tiene relación con el siguiente, esto es normal porque estamos viendo únicamente la última capa.

---

## Ejercicio 4 - Exploración de Capas de Atención

### ¿Qué significa la visualización que acabas de ver?

La gráfica muestra **relaciones de atención** entre palabras:
- **Tokens a la izquierda**: "Desde" qué palabra se mira
- **Tokens a la derecha**: "Hacia" qué palabra se atiende
- **Líneas gruesas/oscuras**: Alta atención
- **Líneas finas/claras**: Baja atención

Por ejemplo, si "banco" tiene una línea gruesa hacia "dinero", significa que el modelo considera que "dinero" es muy relevante para entender el significado de "banco".

Esto es una visualización de la **última capa (capa 11)**. Pero BERT tiene **12 capas** (0 a 11), y cada una aprende cosas diferentes:

```
Entrada → Capa 0 → Capa 1 → ... → Capa 11 → Salida
```

| Capas | Qué aprenden |
|-------|--------------|
| **0-3 (Tempranas)** | Sintaxis: posición, estructura gramatical |
| **4-8 (Medias)** | Relaciones: dependencias entre palabras |
| **9-11 (Finales)** | Semántica: significado contextual profundo |

### Qué hacer:

Modificar **analyze_attention** para ver **diferentes capas** y comparar cómo cambia la atención:
- **Capa 3** (temprana)
- **Capa 6** (media)
- **Capa 11** (final)

¿En qué capa el modelo empieza a "entender" que "banco" está relacionado con "dinero"?

## Paso 1: modifica analyze_attention

In [ ]:
def analyze_attention(sentence, layer=None):
    """
    Extrae la atención de una frase.
    
    Parámetros:
    - sentence: frase a analizar
    - layer: número de capa a visualizar (0-11)
    
    Retorna:
    - tokens: lista de tokens
    - attention: tupla con las matrices de atención de todas las capas
    """
    inputs = tokenizer(sentence, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # ========== TODO: Cambia el siguiente código ==========
    attention = None #TODO: Extraer la atención de la capa especificada (o la última si layer=None)
    
    print(f"Frase: {sentence}")
    print(f"Tokens: {tokens}\n")

    plot_attention(tokens, attention)
    
    return tokens, attention

### Paso 2: Visualiza la capa 3 (temprana)

La capa 3 (temprana) se enfoca más en **sintaxis** (palabras cercanas, estructura gramatical).

In [ ]:
# ========== TODO: Visualiza la capa 3 (temprana) ==========
sentence_test = "Necesito dinero del banco para comprar comida"

analyze_attention()#TODO: Visualiza la capa 3
#¿Qué observas? ¿La palabra "banco" atiende más a palabras cercanas o lejanas?

### Paso 3: Visualiza la capa 6 (media)

La capa 6 (media) balancea **sintaxis y semántica** (empieza a entender relaciones de significado).

In [ ]:
# ========== TODO: Visualiza la capa 6 (media) ==========

analyze_attention()#TODO: Visualiza la capa 6
#¿Qué palabras tienen más atención desde "banco"? ¿Son palabras relevantes para entender que "banco" = institución financiera?

### Paso 4: Visualiza la capa 11 (final)

La capa 11 (final) se enfoca en **semántica profunda** (entiende el significado completo de "banco" según el contexto).

In [ ]:
# ========== TODO: Visualiza la capa 11 (final) ==========

analyze_attention()#TODO: Visualiza la capa 11

---

## Ejercicio 5 - Cabezas de Atención (Attention Heads)

Hasta ahora hemos visto **capas** (layers), pero hay otro nivel de detalle: las **cabezas de atención** (attention heads).

- BERT tiene **12 capas** y cada capa tiene **12 cabezas**
- Total: 144 matrices de atención diferentes
- Cada cabeza aprende patrones distintos:
  - Algunas se especializan en sintaxis (sujeto-verbo)
  - Otras en semántica (palabras relacionadas por significado)
  - Otras en posición (palabras cercanas)

En el ejercicio anterior usábamos `.mean(dim=0)` para observar las capas, lo que **promedia todas las cabezas**. Ahora veremos cabezas individuales.

In [ ]:
def plot_attention_head(tokens, attention, head=0):
    """
    Visualiza UNA cabeza de atención específica.

    Parámetros:
    - head: número de cabeza (0-11)
    """
    # ==========  Extraer atención de UNA cabeza específica (no promedio) ========== 
    att_matrix = None

    #Filtrar tokens especiales
    valid_indices = [i for i, t in enumerate(tokens) if t not in ["[CLS]", "[SEP]"]]
    filtered_tokens = [tokens[i] for i in valid_indices]

    fig, ax = plt.subplots(figsize=(10, 8))

    n_tokens = len(filtered_tokens)
    left_x, right_x = 0.2, 0.8
    y_positions = np.linspace(0.9, 0.1, n_tokens)

    for idx_i, i in enumerate(valid_indices):
        for idx_j, j in enumerate(valid_indices):
            if i == j:
                continue
            weight = att_matrix[i, j]
            weight = weight / att_matrix.max()
            if weight > 0.05:
                ax.plot([left_x, right_x],
                       [y_positions[idx_i], y_positions[idx_j]],
                       color='blue', alpha=weight, linewidth=weight*5)

    for idx, token in enumerate(filtered_tokens):
        ax.text(left_x - 0.05, y_positions[idx], token, ha='right', va='center', fontsize=11, fontweight='bold')
        ax.text(right_x + 0.05, y_positions[idx], token, ha='left', va='center', fontsize=11, fontweight='bold')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(f'Capa 11 - Cabeza {head}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

### Encuentra la cabeza "banco-dinero"

**Qué hay que hacer**: En la capa 11, busca qué cabeza (0-11) tiene la conexión más fuerte entre "banco" y "dinero".

Prueba diferentes valores de `head` y observa cuál muestra una línea más gruesa conectando estas dos palabras.

In [ ]:
#Primero obtenemos la atención de la frase
sentence = "Necesito dinero del banco"
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
attention_layer_11 = outputs.attentions[11]

# ========== TODO: Cambia el número de head (0-11) hasta encontrar la mejor ==========
plot_attention_head(tokens, attention_layer_11, head=0)

**Importante**, en uso normal, el texto pasa por las 12 capas secuencialmente. Cada capa procesa la salida de la anterior y la refina. La salida final (capa 11) es la que usa el modelo para sus predicciones.

---

## Parte 4 - Capacidades de los Transformers

Los transformers son **modelos versátiles** que pueden resolver muchas tareas diferentes solo cambiando la "cabeza" de salida.

Hasta ahora vimos cómo funcionan por dentro (atención, capas). Ahora veremos **qué pueden hacer en la práctica**.

### Tareas comunes de transformers:

1. **Análisis de sentimiento**: ¿La frase es positiva o negativa?
2. **Responder preguntas**: Dado un contexto, responder preguntas
3. **Completar texto**: Predecir palabras faltantes (fill-mask)
4. **Generación de texto**: Continuar una frase
5. **Traducción**: Convertir entre idiomas
6. **Resumen**: Condensar textos largos

---

### Ejemplo 1: Análisis de Sentimiento

**¿Esta frase es positiva o negativa?**

In [ ]:
from transformers import pipeline

#Crear un analizador de sentimiento
sentiment_analyzer = pipeline("sentiment-analysis")

#Probar con diferentes frases, en este caso, este modelo trabaja en inglés
frases_test = [
    "I love this movie, it's fantastic!",
    "I hate waking up early on Mondays",
    "The exam was difficult but I passed it"
]

for frase in frases_test:
    resultado = sentiment_analyzer(frase)[0]
    print(f"Frase: {frase}")
    print(f"→ {resultado['label']} (confianza: {resultado['score']:.2%})")
    print("-" * 60)

---

### Ejemplo 2: Responder Preguntas

**Dado un contexto, el modelo responde preguntas sobre él**

In [ ]:
#Crear un modelo de preguntas y respuestas (1 línea)
qa_model = pipeline("question-answering")

#Contexto sobre el que preguntar, en inglés porque el modelo es monolingüe
contexto = """
James Sunderland receives a mysterious letter from his wife, Mary, 
inviting him to their "special place" in Silent Hill, 
despite her dying three years prior. Confused, James visits the eerie, 
foggy town, where he encounters distorted monsters and other lost souls, 
with the environment acting as a terrifying reflection of his own guilt and repressed trauma.
"""

#Diferentes preguntas
preguntas = [
    "Who is James's wife?",
    "Where did James go?",
    "What did James find in Silent Hill?",
]

for pregunta in preguntas:
    respuesta = qa_model(question=pregunta, context=contexto)
    print(f"Pregunta: {pregunta}")
    print(f"→ Respuesta: {respuesta['answer']} (confianza: {respuesta['score']:.2%})")
    print("-" * 60)

---

## Ejercicio 5 - Predicción de Palabras (Fill-Mask)

Además de analizar sentimientos y responder preguntas, los transformers también pueden **predecir palabras faltantes** en una frase.

Esta tarea se llama **fill-mask** y es muy útil para:
- Autocompletar texto
- Corregir errores
- Sugerir palabras en editores

Los modelos transformer pueden predecir qué palabra debería ir en el lugar de `[MASK]` observando el contexto completo de la frase.

### Ejemplo con BERT Multilingüe

BERT es un modelo **bidireccional** (lee el texto en ambas direcciones) diseñado para tareas de comprensión, perfecto para fill-mask.

In [ ]:
from transformers import pipeline

#Crear el pipeline de fill-mask con BERT
unmasker_bert = pipeline("fill-mask", model="bert-base-multilingual-cased")

#Probar con diferentes frases
test_sentences = [
    "Necesito dinero del [MASK]",
    "Me siento en el [MASK] del parque",
    "El [MASK] es el mejor amigo del hombre"
]

print("=" * 60)
print("MODELO: BERT Multilingüe (bert-base-multilingual-cased)")
print("=" * 60)

for sentence in test_sentences:
    print(f"\nFrase: {sentence}")
    results = unmasker_bert(sentence)
    print("Top 3 predicciones:")
    for i, r in enumerate(results[:3], 1):
        print(f"  {i}. {r['token_str']} (score: {r['score']:.4f})")

---

### Qué hay que hacer: Busca un modelo diferente

**Tarea**: Prueba con alguno de los modelos sugeridos y busca un modelo **diferente** (puede ser de otro idioma, más pequeño, o especializado) ycomprieba cuál genera mejores salidas.

**Sugerencias de modelos**:
- **RoBERTa** (inglés): `roberta-base`
- **BETO** (español): `dccuchile/bert-base-spanish-wwm-cased`
- **CamemBERT** (francés): `camembert-base`
- **DistilBERT** (ligero): `distilbert-base-multilingual-cased`
- Busca uno en: [Hugging Face - Fill Mask Models](https://huggingface.co/models?pipeline_tag=fill-mask)

**Importante**: 
- Algunos modelos usan `<mask>` en lugar de `[MASK]`
- Algunos solo funcionan en un idioma específico
- Si da error, prueba otro modelo

**Modifica el código siguiente** con tu modelo elegido.

In [ ]:
# ========== TODO: Reemplaza con el modelo que quieras probar ==========
my_model = None  #Ej: "roberta-base", "dccuchile/bert-base-spanish-wwm-cased"

# ========== TODO: Cambia "[MASK]" si tu modelo usa otro token (ej: "<mask>") ==========
mask_token = "[MASK]"  #Algunos modelos usan "<mask>" o "<<mask>>"

if my_model is not None:
    try:
        #Crear pipeline con tu modelo
        unmasker_custom = pipeline("fill-mask", model=my_model)
        
        # ========== TODO: Añade frases para que tu modelo las complete (ajusta si tu modelo es de otro idioma) ==========
        test_sentences_custom = [
            f"... {mask_token} ...", #TODO
            f"... {mask_token} ...", #TODO
            f"... {mask_token} ...", #TODO
        ]
        
        print("=" * 60)
        print(f"MODELO: {my_model}")
        print("=" * 60)
        
        for sentence in test_sentences_custom:
            print(f"\nFrase: {sentence}")
            results = unmasker_custom(sentence)
            print("Top 3 predicciones:")
            for i, r in enumerate(results[:3], 1):
                print(f"  {i}. {r['token_str']} (score: {r['score']:.4f})")
    
    except Exception as e:
        print(f"Error al cargar el modelo: {e}")
        print("Prueba con otro modelo o verifica el nombre.")
else:
    print("Debes especificar un modelo en 'my_model' arriba.")